# 第2章  变化的直觉 —— 斜率就是速度

> **本章目标**：建立导数=瞬时变化率=切线斜率的几何直觉。用数值实验验证割线逼近切线的过程，理解h不能太小的原因（浮点舍入误差），制作收敛动画。
> **前置知识**：第1章（函数、np.linspace、matplotlib绘图模板）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
print('环境就绪')


---
## 2.1  割线斜率 —— 平均速度的几何翻译

割线穿过曲线上两点(a,f(a))和(b,f(b))，其斜率=(f(b)-f(a))/(b-a)=平均变化率。

📐 **割线（Secant Line）与平均变化率**：区间[a,b]上的平均变化率。

🔗 **AI连接**：每个epoch的loss下降量就是该epoch的平均变化率。

In [ ]:
import numpy as np

def f(x):
    return x ** 2

def secant_slope(f, a, b):
    return (f(b) - f(a)) / (b - a)

a = 1.0
b_values = [3.0, 2.0, 1.5, 1.1, 1.01]

print(f'f(x) = x^2, a = {a}')
for b in b_values:
    slope = secant_slope(f, a, b)
    print(f'  [{a:.2f}, {b:<5.2f}]  slope = {slope:.4f}')
print(f'\n导数 f\'(1) = 2*1 = 2.0  <-- 割线斜率正在逼近!')


---
## 2.2  让两点无限逼近 —— 浮点精度的边界

固定a=2，让h越来越小——但h小到10^-8以下时，浮点舍入误差主导，斜率反而失真。

📐 **瞬时变化率**：f'(a)=lim(h->0)[f(a+h)-f(a)]/h。几何上=切线斜率。

🔗 **AI连接**：PyTorch的autograd用链式法则精确计算梯度，完全避开h选择问题（第13-14章）。

In [ ]:
import numpy as np

def f(x):
    return x ** 2

def secant_slope_h(f, a, h):
    return (f(a + h) - f(a)) / h

a = 2.0
true_derivative = 2 * a  # f'(2) = 4

print(f'f(x)=x^2 at x={a}, true f\'({a}) = {true_derivative}')
print(f'{"h":<14} {"slope":<18} {"error":<12} {"status"}')
print('-' * 56)

for h in [1.0, 1e-2, 1e-4, 1e-5, 1e-6, 1e-8, 1e-10, 1e-12, 1e-14, 1e-16]:
    slope = secant_slope_h(f, a, h)
    error = abs(slope - true_derivative)
    if error < 1e-6:
        status = 'OK'
    elif error < 1e-3:
        status = 'good'
    elif error < 0.1:
        status = 'degrading'
    else:
        status = 'BROKEN'
    print(f'h={h:<12.0e} slope={slope:<16.10f} err={error:<10.2e} {status}')


---
## 2.3  动画演示 —— 让割线活起来

60帧动画：红色割线绕固定点旋转，随着h->0缓缓躺平成切线。

🔗 **AI连接**：FuncAnimation在第12章展示梯度下降轨迹，在第24章对比优化器。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

def f(x):
    return x ** 2

a = 2.0
x_range = np.linspace(0, 4, 400)
h_start, h_end = 2.0, 0.02
n_frames = 60
h_values = np.logspace(np.log10(h_start), np.log10(h_end), n_frames)

fig, ax = plt.subplots(figsize=(8, 6))
ax.set_xlim(0, 4); ax.set_ylim(0, 16)
ax.set_xlabel('x'); ax.set_ylabel('f(x)')
ax.grid(alpha=0.3)
ax.plot(x_range, f(x_range), 'b-', linewidth=2, label='f(x)=x^2')
ax.plot(a, f(a), 'ro', markersize=8, zorder=5, label=f'切点({a},{f(a)})')

secant_line, = ax.plot([], [], 'r-', linewidth=1.5, alpha=0.8, label='割线')
movable_point, = ax.plot([], [], 'go', markersize=6)
info_text = ax.text(0.05, 0.92, '', transform=ax.transAxes, fontsize=10,
                    fontfamily='monospace', bbox=dict(boxstyle='round', alpha=0.8))
ax.legend(loc='upper left')

def init():
    secant_line.set_data([], [])
    movable_point.set_data([], [])
    info_text.set_text('')
    return secant_line, movable_point, info_text

def animate(frame):
    h = h_values[frame]
    b = a + h
    slope = (f(b) - f(a)) / h
    x_line = np.array([0, 4])
    y_line = f(a) + slope * (x_line - a)
    secant_line.set_data(x_line, y_line)
    movable_point.set_data([b], [f(b)])
    info_text.set_text(f'h={h:.3f}\nslope={slope:.4f}\ntarget={2*a:.1f}')
    return secant_line, movable_point, info_text

ani = animation.FuncAnimation(fig, animate, init_func=init,
                              frames=n_frames, interval=80, blit=True)
ani.save('secant_to_tangent.gif', writer='pillow', fps=12, dpi=90)
print('动画已保存: secant_to_tangent.gif')
plt.show()


---
## 2.4  总结：从割线到切线

1. 割线=平均变化率  2. 缩小区间  3. 极限位置=切线  4. 切线斜率=导数

📐 **导数（Derivative）**：f'(a)=lim(h->0)[f(a+h)-f(a)]/h。几何上=切线斜率。

两个工程边界：h太大->近似误差，h太小->舍入误差。PyTorch的autograd完全避开此问题。

---
## 习题

> 在下方新建代码单元格作答。

1. （概念）割线斜率和切线斜率的本质区别是什么？用平均速度和瞬时速度比喻。
2. （概念）h=1e-5和h=1e-15哪个给出的导数近似更准确？用近似误差和舍入误差解释。
3. （代码）对f(x)=x^3在x=2处重复实验（理论导数=12），打印h从1.0到1e-14的割线斜率。
4. （代码）修改动画脚本，切点改为x=1，函数改为f(x)=x^3，保存新GIF。

---

> **章末钩子**：你现在能用缩小区间的方法逼近任何函数的瞬时变化率了。但每次做实验太慢——有没有公式可以直接秒算？
> 预览：第10章——导数，系统学习AI常用函数的解析导数公式。

> **提示**：完成后运行 Kernel -> Restart & Run All 验证。